                                            Artifacts Building for Recommendation System

## Objective
To preprocess raw transactional data and build reusable artifacts that capture item relationships, user behavior, and contextual patterns for fast and scalable recommendation.
## Data Sources
The system uses three main datasets:

Raw transactional data (main_data.csv)
Item catalog (item_catalog.csv)
Category embedding lookup (category_embedding_lookup)
## Data Cleaning and Normalization

Converts itemId and customerId into numeric format
Cleans item names and categories
Parses orderDate into datetime format
Removes invalid or missing rows
Computes additional features such as weekOfMonth
## Lookup Table Construction

itemId → category mapping
itemId → itemName mapping
category → embedding vector mapping

These lookups enable fast access during recommendation.
## Basket Table Creation
Raw transactions are grouped into baskets using transactionId:

Each basket contains a list of items
Includes contextual information such as season, time slot, and holiday flags

Purpose:
To represent user purchase sessions in structured form.
## Association Rule Mining

Uses FP-Growth algorithm to find frequent itemsets
Generates association rules based on confidence
Captures relationships such as “if A then B”

Purpose:
To identify strong co-purchase patterns for candidate generation.

## Item Co-purchase Counter

Counts how frequently item pairs occur together
Stores directional relationships between items

Purpose:
To support collaborative filtering signals.
## Context-Based Popularity Counter

Builds mapping: (context → item frequency)
Context includes season, holiday, festival, time slot, and week

Purpose:
To enable context-aware recommendations.
## User Behavior Counters

User → item frequency
User → category frequency

Purpose:
To capture user preferences and personalization signals.
## Date Context Lookup

Maps each date to contextual attributes:
season
holiday
festival

Purpose:
To quickly retrieve context based on date during inference.

## Artifact Storage
All computed artifacts are saved as serialized files:

Association rules
Item pair counters
Context counters
User history counters
Lookup tables
Embedding mappings

Purpose:
To reuse precomputed data during candidate generation without recomputation.
Benefits

Reduces runtime computation during recommendation
Enables fast candidate generation
Captures multiple behavioral signals (co-purchase, context, user preference)
Improves scalability of the system
Provides reusable components across pipelines
## Role in Recommendation System
These artifacts are used in Stage 1 (candidate generation):

Generate relevant candidate items
Filter large item space into a manageable subset
Provide signals for ranking models
## Recommendation Flow (Stage 1)

Input: user context and basket
Use artifacts to generate candidate items
Combine signals from co-purchase, context, and user history
Output: candidate pool for ranking stage

 IMPORTS


In [2]:

import json
import pickle
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

 PATHS


In [2]:

DATA_DIR = Path(r"D:\recommendation_item_API\data")

RAW_FILE = DATA_DIR / "main_data.csv"
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog.csv"
CATEGORY_LOOKUP_FILE = DATA_DIR / "category_embedding_lookup_from_customer_history.csv"

ARTIFACT_DIR = DATA_DIR / "stage1_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw file exists:", RAW_FILE.exists())
print("Catalog exists:", ITEM_CATALOG_FILE.exists())
print("Category lookup exists:", CATEGORY_LOOKUP_FILE.exists())

Raw file exists: True
Catalog exists: True
Category lookup exists: True


LOAD DATA


In [3]:

raw_df = pd.read_csv(RAW_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)
cat_lookup_df = pd.read_csv(CATEGORY_LOOKUP_FILE)

raw_df.columns = [c.strip() for c in raw_df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]
cat_lookup_df.columns = [c.strip() for c in cat_lookup_df.columns]

display(raw_df.head())
display(item_catalog_df.head())
display(cat_lookup_df.head())

,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isFestival,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0


,category,cat_emb_0,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,category_embedding
0,Bakery-Bread,30.500512,-0.312345,-1.336538,0.171504,1.265434,-0.221737,-2.422268,-1.024791,"[30.500512, -0.312345, -1.336538, 0.171504, 1...."
1,Beverage-Carbonated,30.793406,0.582054,2.320642,-1.194838,0.826971,-1.544423,1.249234,-0.971547,"[30.793406, 0.582054, 2.320642, -1.194838, 0.8..."
2,Beverage-Hot,29.132634,0.224027,0.067448,-0.667184,-0.053071,0.364866,0.603695,0.191561,"[29.132634, 0.224027, 0.067448, -0.667184, -0...."
3,Beverage-Juice,31.321391,0.378736,0.439785,-0.637943,0.784723,-1.583087,-2.383535,2.169788,"[31.321391, 0.378736, 0.439785, -0.637943, 0.7..."
4,Beverage-Water,22.344446,0.596863,0.331393,-0.024937,0.778449,0.414786,-0.005348,-0.207390,"[22.344446, 0.596863, 0.331393, -0.024937, 0.7..."


HELPERS


In [4]:

def normalize_text(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_category(cat):
    if pd.isna(cat):
        return cat
    cat = normalize_text(cat)
    parts = [p.strip() for p in cat.split("-")]
    parts = [p.capitalize() if p else p for p in parts]
    return "-".join(parts)

def week_of_month(dt):
    return ((dt.day - 1) // 7) + 1

CLEAN RAW DATA


In [5]:

raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")
raw_df["customerId"] = pd.to_numeric(raw_df["customerId"], errors="coerce")
raw_df["category"] = raw_df["category"].apply(normalize_category)
raw_df["itemName"] = raw_df["itemName"].apply(normalize_text)

raw_df["orderDate"] = pd.to_datetime(raw_df["orderDate"], dayfirst=True, errors="coerce")
raw_df = raw_df.dropna(subset=["itemId", "customerId", "orderDate"]).copy()

raw_df["itemId"] = raw_df["itemId"].astype(int)
raw_df["customerId"] = raw_df["customerId"].astype(int)
raw_df["weekOfMonth"] = raw_df["orderDate"].apply(week_of_month)

print("Cleaned raw shape:", raw_df.shape)
display(raw_df.head())

Cleaned raw shape: (16108, 33)


,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month,weekOfMonth
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,Winter,Morning,Weekday,Wednesday,11,1,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,Winter,Morning,Weekday,Wednesday,11,1,1


 LOOKUPS


In [6]:

item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], errors="coerce")
item_catalog_df = item_catalog_df.dropna(subset=["itemId", "category"]).copy()
item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["category"] = item_catalog_df["category"].apply(normalize_category)

item_to_category = dict(zip(item_catalog_df["itemId"], item_catalog_df["category"]))
item_to_name = dict(zip(item_catalog_df["itemId"], item_catalog_df["itemName"]))

cat_lookup_df["category"] = cat_lookup_df["category"].apply(normalize_category)
cat_lookup_df["category_embedding_vec"] = cat_lookup_df["category_embedding"].apply(json.loads)

category_to_vector = {
    row["category"]: np.array(row["category_embedding_vec"], dtype=float)
    for _, row in cat_lookup_df.iterrows()
}

print("item_to_category:", len(item_to_category))
print("category_to_vector:", len(category_to_vector))

item_to_category: 229
category_to_vector: 43


 BASKET TABLE


In [7]:

basket_df = (
    raw_df
    .sort_values(["transactionId", "orderDate"])
    .groupby("transactionId")
    .agg({
        "customerId": "first",
        "orderDate": "first",
        "season": "first",
        "isHoliday": "max",
        "isFestival": "max",
        "timeSlot": "first",
        "weekOfMonth": "first",
        "itemId": lambda x: list(map(int, x.tolist()))
    })
    .reset_index()
)

basket_df.rename(columns={"itemId": "item_ids"}, inplace=True)

print("Basket shape:", basket_df.shape)
display(basket_df.head())

Basket shape: (2426, 9)


,transactionId,customerId,orderDate,season,isHoliday,isFestival,timeSlot,weekOfMonth,item_ids
0,1,23561,2025-01-01 07:29:00,Winter,0,0,Morning,1,"[7075, 27, 6814]"
1,2,23569,2025-01-01 11:33:00,Winter,0,0,Morning,1,"[25474, 15131, 13786, 7017, 6815, 31849, 13352]"
2,3,23527,2025-01-01 12:39:00,Winter,0,0,Afternoon,1,"[32441, 2306, 13922, 2364, 952, 30743, 32269]"
3,4,23433,2025-01-01 12:46:00,Winter,0,0,Afternoon,1,"[878, 704, 15104, 31328, 14964]"
4,5,23494,2025-01-01 13:21:00,Winter,0,0,Afternoon,1,"[15180, 2220, 2364, 13793, 708, 2043, 12655]"


ASSOCIATION RULES


In [9]:

basket_items_for_rules = basket_df["item_ids"].apply(
    lambda x: [str(v) for v in sorted(set(x))]
).tolist()

te = TransactionEncoder()
basket_matrix = te.fit(basket_items_for_rules).transform(basket_items_for_rules)
basket_matrix_df = pd.DataFrame(basket_matrix, columns=te.columns_)

freq_items = fpgrowth(
    basket_matrix_df,
    min_support=0.005,
    use_colnames=True,
    max_len=3
)

freq_items["itemset_len"] = freq_items["itemsets"].apply(len)

print("Frequent itemsets shape:", freq_items.shape)
display(freq_items.head())

# গুরুত্বপূর্ণ, এখানে full freq_items দিতে হবে
rules_df = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.10
)

# চাইলে পরে rule filter করতে পারো
rules_df["antecedent_len"] = rules_df["antecedents"].apply(len)
rules_df["consequent_len"] = rules_df["consequents"].apply(len)

rules_df = rules_df[
    (rules_df["antecedent_len"] >= 1) &
    (rules_df["consequent_len"] >= 1)
].copy()

rules_df["antecedents"] = rules_df["antecedents"].apply(lambda s: sorted(list(s)))
rules_df["consequents"] = rules_df["consequents"].apply(lambda s: sorted(list(s)))

print("Rules shape:", rules_df.shape)
display(rules_df.head())

Frequent itemsets shape: (704, 3)


,support,itemsets,itemset_len
0,0.040396,(6814),1
1,0.037510,(27),1
2,0.023495,(7075),1
3,0.087799,(13352),1
4,0.079555,(25474),1


Rules shape: (713, 16)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len
0,[6814],[3831],0.040396,0.044930,0.006183,0.153061,3.406665,1.0,0.004368,1.127673,0.736197,0.078125,0.113218,0.145338,1,1
1,[3831],[6814],0.044930,0.040396,0.006183,0.137615,3.406665,1.0,0.004368,1.112733,0.739692,0.078125,0.101312,0.145338,1,1
2,[6814],[13352],0.040396,0.087799,0.007007,0.173469,1.975759,1.0,0.003461,1.103651,0.514655,0.057823,0.093916,0.126641,1,1
3,[6814],[14034],0.040396,0.098516,0.005359,0.132653,1.346512,1.0,0.001379,1.039358,0.268173,0.040123,0.037868,0.093523,1,1
4,[6814],[27],0.040396,0.037510,0.007832,0.193878,5.168648,1.0,0.006317,1.193975,0.840477,0.111765,0.162461,0.201334,1,1


CO PURCHASE COUNTER


In [10]:

item_pair_counter = Counter()

for _, row in basket_df.iterrows():
    items = sorted(set(row["item_ids"]))
    n = len(items)

    for i in range(n):
        for j in range(i + 1, n):
            a = items[i]
            b = items[j]
            item_pair_counter[(a, b)] += 1
            item_pair_counter[(b, a)] += 1

print("Directional item pairs:", len(item_pair_counter))

Directional item pairs: 30024


CONTEXT POPULARITY COUNTER


In [11]:

context_item_counter = Counter()

for _, row in raw_df.iterrows():
    context_key = (
        row["season"],
        int(row["isHoliday"]),
        int(row["isFestival"]),
        row["timeSlot"],
        int(row["weekOfMonth"])
    )
    context_item_counter[(context_key, int(row["itemId"]))] += 1

print("Context item pairs:", len(context_item_counter))

Context item pairs: 6591


USER HISTORY COUNTERS


In [12]:

user_item_counter = Counter()
user_category_counter = Counter()

for _, row in raw_df.iterrows():
    user_id = int(row["customerId"])
    item_id = int(row["itemId"])
    category = row["category"]

    user_item_counter[(user_id, item_id)] += 1
    user_category_counter[(user_id, category)] += 1

print("User item history size:", len(user_item_counter))
print("User category history size:", len(user_category_counter))

User item history size: 7275
User category history size: 3115


 DATE CONTEXT LOOKUP


In [13]:

date_context_df = (
    raw_df
    .assign(orderDay=raw_df["orderDate"].dt.date.astype(str))
    .groupby("orderDay")
    .agg({
        "season": lambda x: x.mode().iat[0] if len(x.mode()) > 0 else x.iloc[0],
        "isHoliday": "max",
        "isFestival": "max"
    })
    .reset_index()
)

date_context_lookup = {
    row["orderDay"]: {
        "season": row["season"],
        "isHoliday": int(row["isHoliday"]),
        "isFestival": int(row["isFestival"])
    }
    for _, row in date_context_df.iterrows()
}

print("Date context entries:", len(date_context_lookup))
display(date_context_df.head())

Date context entries: 144


,orderDay,season,isHoliday,isFestival
0,2025-01-01,Winter,0,0
1,2025-01-02,Winter,1,0
2,2025-01-03,Summer,1,1
3,2025-01-04,Summer,0,1
4,2025-01-05,Summer,1,0


 CUSTOMER DEFAULT TIMESLOT


In [14]:

customer_default_timeslot_df = (
    raw_df.groupby("customerId")["timeSlot"]
    .agg(lambda x: x.mode().iat[0] if len(x.mode()) > 0 else x.iloc[0])
    .reset_index()
)

customer_default_timeslot = dict(
    zip(customer_default_timeslot_df["customerId"], customer_default_timeslot_df["timeSlot"])
)

print("Customer default timeslot size:", len(customer_default_timeslot))
display(customer_default_timeslot_df.head())

Customer default timeslot size: 137


,customerId,timeSlot
0,4959,Evening
1,23380,Evening
2,23381,Morning
3,23383,Night
4,23385,Night


SAVE ARTIFACTS


In [15]:

rules_file = ARTIFACT_DIR / "association_rules.pkl"
pair_file = ARTIFACT_DIR / "item_pair_counter.pkl"
context_file = ARTIFACT_DIR / "context_item_counter.pkl"
user_item_file = ARTIFACT_DIR / "user_item_counter.pkl"
user_cat_file = ARTIFACT_DIR / "user_category_counter.pkl"
item_to_cat_file = ARTIFACT_DIR / "item_to_category.pkl"
item_to_name_file = ARTIFACT_DIR / "item_to_name.pkl"
cat_vec_file = ARTIFACT_DIR / "category_to_vector.pkl"
date_ctx_file = ARTIFACT_DIR / "date_context_lookup.pkl"
cust_ts_file = ARTIFACT_DIR / "customer_default_timeslot.pkl"

with open(rules_file, "wb") as f:
    pickle.dump(rules_df, f)

with open(pair_file, "wb") as f:
    pickle.dump(item_pair_counter, f)

with open(context_file, "wb") as f:
    pickle.dump(context_item_counter, f)

with open(user_item_file, "wb") as f:
    pickle.dump(user_item_counter, f)

with open(user_cat_file, "wb") as f:
    pickle.dump(user_category_counter, f)

with open(item_to_cat_file, "wb") as f:
    pickle.dump(item_to_category, f)

with open(item_to_name_file, "wb") as f:
    pickle.dump(item_to_name, f)

with open(cat_vec_file, "wb") as f:
    pickle.dump(category_to_vector, f)

with open(date_ctx_file, "wb") as f:
    pickle.dump(date_context_lookup, f)

with open(cust_ts_file, "wb") as f:
    pickle.dump(customer_default_timeslot, f)

print("Stage 1 artifacts saved.")

Stage 1 artifacts saved.
